In [4]:
import torch
import random
import numpy as np
from unakin.llm._data.datasources.train_sample import LMBatchMetadata, LMTrainingBatch
from typing import Dict, Any, List, Union
from dataclasses import dataclass

random.seed(0)
np.random.seed(0)
torch.manual_seed(0)


def tensor_mask_sum_acc(
    values: torch.Tensor,
    mask: torch.Tensor,
):
  
  if len(values.shape) == 2 and values.shape[0] == 1:
    values = values.squeeze()

  if len(mask.shape) == 2 and mask.shape[0] == 1:
    mask = mask.squeeze()


  # Initialize tensors for sum and count in float32 to prevent overflow.
  # for 256K sequences, we need 8MB of memory per tensor
  # therefore a masked tensor requires around 16MB of memory for 256K sequences.
  # 6 such constructs will require around 100MBs of memory
  sum_vals = torch.zeros(mask.max() + 1, device=values.device, dtype=torch.float32)
  counts = torch.zeros(mask.max() + 1, device=values.device, dtype=torch.float32)

  # Use torch.scatter_add() to sum values based on mask positions
  sum_vals.scatter_add_(0, mask, values)
  # Count occurrences of each position in the mask
  counts.scatter_add_(0, mask, torch.ones_like(values))

  return {
    "tensor": sum_vals.unsqueeze(0),
    "mask": counts.unsqueeze(0)
  }

def is_masked_tensor_dict(d: Any) -> bool:
  return "mask" in d and "tensor" in d

def group_by_keys_recursive(dicts: List[dict], results = None) -> dict:

  if results is None:
    results = {}

  for d in dicts:
    for key, value in d.items():

      if isinstance(value, dict):
        results[key] = group_by_keys_recursive([value], results.get(key, None))
      else:

        if key in results:
          results[key].append(value)
        else:
          results[key] = [value]

  return results

def reduce_multiple_stats_steps(stats: List[dict]) -> dict:
  # this is going to be a dict with multiple keys
  flat_stats = group_by_keys_recursive(stats)
  for k, v in flat_stats.items():
      if is_masked_tensor_dict(v):
        v["mask"] = torch.nested.as_nested_tensor(v["mask"])
        v["tensor"] = torch.nested.as_nested_tensor(v["tensor"])

  # prepare everything for global-reduce
  for k, v in flat_stats.items():
    if is_masked_tensor_dict(v):
      v["mask"] = torch.nested.to_padded_tensor(v["mask"], padding=0)
      v["tensor"] = torch.nested.to_padded_tensor(v["tensor"], padding=0)
      
      v["mask"].sum(dim=0)      
      v["tensor"].sum(dim=0)
      # v["mean"] = v["tensor"] / v["mask"]
    else:
      flat_stats[k] = torch.stack(v).mean()
  
  return flat_stats

####################################################

@dataclass
class BatchTest:
  positional_ids: torch.Tensor
  datasource_indices: torch.Tensor
  output_tokens: torch.Tensor

  def __getitem__(self, item: Union[int, slice]):
    sliced_pos_ids = self.positional_ids[item]
    sliced_ds_indices = self.datasource_indices[item]
    sliced_out_tokens = self.output_tokens[item]
    return BatchTest(sliced_pos_ids, sliced_ds_indices, sliced_out_tokens)

#####################################################

def process_train_logging_metrics(
  batch: BatchTest,
  preds: torch.Tensor,
  loss:  torch.Tensor,
  metric_prefix: str = "train"
) -> dict:

  def PREFIX_VAR(name: str) -> str:
    return f"{metric_prefix}_{name}"

  ppl_per_token = 2 ** loss
  precision_per_token = (preds.argmax(dim=-1) == batch.output_tokens).type(torch.float32)
  
  pos_ppl_per_token       = tensor_mask_sum_acc(ppl_per_token, batch.positional_ids)
  pos_precision_per_token = tensor_mask_sum_acc(precision_per_token, batch.positional_ids)

  ds_ppl_per_token        = tensor_mask_sum_acc(ppl_per_token, batch.datasource_indices)
  ds_precision_per_token  = tensor_mask_sum_acc(precision_per_token, batch.datasource_indices)

  metrics = {
    PREFIX_VAR("perplexity"): ppl_per_token.mean(),
    PREFIX_VAR("precision"): precision_per_token.mean(),

    PREFIX_VAR("pos_perplexity"): pos_ppl_per_token,
    PREFIX_VAR("pos_precision"): pos_precision_per_token,

    PREFIX_VAR("ds_perplexity"): ds_ppl_per_token,
    PREFIX_VAR("ds_precision"): ds_precision_per_token,
  }

  return metrics

In [5]:
MAX_LEN = 256

# get random seq_lens
seq_lens = []
while sum(seq_lens) < MAX_LEN:
  seq_lens.append(random.randint(8, 32))
# cut the last seqlen
seq_lens[-1] = MAX_LEN - sum(seq_lens[:-1])

pos_ids = []
for s_len in seq_lens:
  pos_ids.extend(list(range(s_len)))
pos_ids = torch.tensor(pos_ids)

batch_dict = {
  "positional_ids": pos_ids.unsqueeze(0),
  "datasource_indices": torch.randint(0, 15, (MAX_LEN,)).unsqueeze(0),
  "output_tokens": torch.randint(0, 5, (MAX_LEN,)).unsqueeze(0)
}
batch = BatchTest(**batch_dict)

preds = torch.randint(0, 5, (1, MAX_LEN, 5))

loss = torch.randn(MAX_LEN)
loss = torch.abs(loss).unsqueeze(0)

print("LOSSES    : ", loss)
print("POS ids   : ", batch.positional_ids)
print("DS inds   : ", batch.datasource_indices)

LOSSES    :  tensor([[1.9052, 1.6952, 1.0106, 0.7171, 1.1469, 0.0763, 0.4433, 0.5973, 0.9660,
         0.0236, 1.1778, 0.6408, 1.2593, 1.6406, 1.4281, 0.9862, 0.6179, 1.7282,
         0.2725, 1.2994, 1.1093, 1.8811, 1.3008, 0.7654, 0.4777, 1.2307, 0.4958,
         1.0617, 1.2442, 0.5508, 0.5205, 1.5414, 0.0360, 0.8797, 0.9801, 1.6861,
         0.2368, 1.5649, 0.2233, 0.1753, 0.0859, 0.3875, 1.1794, 1.5783, 0.2082,
         0.4852, 0.0437, 0.1160, 0.7978, 0.2425, 0.4761, 0.3796, 0.0114, 0.4512,
         1.0632, 0.2497, 0.0545, 0.8129, 0.9576, 1.2139, 0.5725, 0.0793, 1.1229,
         1.4157, 0.2296, 0.1575, 1.1857, 1.2910, 0.0035, 0.7711, 2.6715, 1.9461,
         2.2148, 1.6746, 0.2388, 1.9177, 0.6798, 2.5549, 1.3455, 1.9527, 0.8377,
         1.4118, 0.4003, 0.6820, 0.8854, 0.3796, 0.4915, 0.4885, 0.3421, 0.4579,
         1.3709, 0.0459, 0.8977, 0.3353, 0.7939, 0.6968, 0.2297, 0.5879, 0.4736,
         0.0257, 0.4650, 0.6946, 0.0423, 0.9025, 0.5860, 0.4640, 1.2195, 2.5317,
         2.5865

In [6]:
reporting_stats_single = []
block_reporting_stats = process_train_logging_metrics(
  batch,
  preds,
  loss
)
reporting_stats_single.append(block_reporting_stats)
local_reporting_stats_single = reduce_multiple_stats_steps(reporting_stats_single)

local_reporting_stats_single

{'train_perplexity': tensor(1.8701),
 'train_precision': tensor(0.2109),
 'train_pos_perplexity': {'tensor': tensor([[[23.7845, 23.0270, 24.0598, 19.4307, 25.4467, 22.3566, 21.2348,
            22.5004, 25.1177, 21.1940, 26.5213, 20.0966, 16.1687, 18.3523,
            21.2024, 29.4590, 13.4063, 18.7890, 18.5307, 19.9818, 13.2156,
             6.4439,  9.6885,  5.8796,  2.6755,  2.4524,  1.0308,  1.0837,
             1.7384,  1.1831,  1.3909,  1.3010]]]),
  'mask': tensor([[[13., 13., 13., 13., 13., 13., 12., 12., 12., 11., 11., 11., 11., 11.,
            11., 11., 10.,  9.,  9.,  8.,  6.,  5.,  5.,  3.,  2.,  2.,  1.,  1.,
             1.,  1.,  1.,  1.]]])},
 'train_pos_precision': {'tensor': tensor([[[4., 0., 2., 4., 1., 1., 4., 2., 2., 4., 2., 2., 6., 6., 4., 1., 1.,
            2., 1., 1., 1., 2., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0.]]]),
  'mask': tensor([[[13., 13., 13., 13., 13., 13., 12., 12., 12., 11., 11., 11., 11., 11.,
            11., 11., 10.,  9.,  9.,  8.,  6.,  5.,  

In [4]:
block_size = 512
N_BLOCKS, should_inc = divmod(MAX_LEN, block_size)
N_BLOCKS = N_BLOCKS + 1 if should_inc else N_BLOCKS

reporting_stats_multi = []
for i in range(N_BLOCKS):
  start = i * block_size
  end = (i + 1) * block_size


  batch_slice = batch[:, start:end]
  preds_slice = preds[:, start:end]
  loss_slice = loss[:, start:end]


  block_reporting_stats_slice = process_train_logging_metrics(
                                          batch_slice,
                                          preds_slice,
                                          loss_slice
                                          )
  reporting_stats_multi.append(block_reporting_stats_slice)


local_reporting_stats_multi = reduce_multiple_stats_steps(reporting_stats_multi)

local_reporting_stats_multi

{'train_perplexity': tensor(1.8701),
 'train_precision': tensor(0.2109),
 'train_pos_perplexity': {'tensor': tensor([[[23.7845, 23.0270, 24.0598, 19.4307, 25.4467, 22.3566, 21.2348,
            22.5004, 25.1177, 21.1940, 26.5213, 20.0966, 16.1687, 18.3523,
            21.2024, 29.4590, 13.4063, 18.7890, 18.5307, 19.9818, 13.2156,
             6.4439,  9.6885,  5.8796,  2.6755,  2.4524,  1.0308,  1.0837,
             1.7384,  1.1831,  1.3909,  1.3010]]]),
  'mask': tensor([[[13., 13., 13., 13., 13., 13., 12., 12., 12., 11., 11., 11., 11., 11.,
            11., 11., 10.,  9.,  9.,  8.,  6.,  5.,  5.,  3.,  2.,  2.,  1.,  1.,
             1.,  1.,  1.,  1.]]])},
 'train_pos_precision': {'tensor': tensor([[[4., 0., 2., 4., 1., 1., 4., 2., 2., 4., 2., 2., 6., 6., 4., 1., 1.,
            2., 1., 1., 1., 2., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0.]]]),
  'mask': tensor([[[13., 13., 13., 13., 13., 13., 12., 12., 12., 11., 11., 11., 11., 11.,
            11., 11., 10.,  9.,  9.,  8.,  6.,  5.,  